In [2]:
import sqlite3
import pandas as pd
import sys
from pathlib import Path

ROOT = Path("../").resolve()
sys.path.append(str(ROOT))
from src.analytics.cashflow_kpis import *

conn = sqlite3.connect("../data/nifty100.db")

In [3]:
pl = pd.read_sql("""
SELECT
company_id,
year,
sales,
operating_profit,
net_profit
FROM profitandloss
""", conn)

cf = pd.read_sql("""
SELECT
company_id,
year,
operating_activity,
investing_activity,
financing_activity
FROM cashflow
""", conn)

conn.close()

In [4]:
df = pl.merge(
    cf,
    on=["company_id", "year"]
)

In [5]:
df["free_cash_flow"] = df.apply(
    lambda x: calculate_free_cash_flow(
        x.operating_activity,
        x.investing_activity
    ),
    axis=1
)

df["cfo_quality"] = df.apply(
    lambda x: calculate_cfo_quality(
        x.operating_activity,
        x.net_profit
    ),
    axis=1
)

df["capex_intensity"] = df.apply(
    lambda x: calculate_capex_intensity(
        x.investing_activity,
        x.sales
    ),
    axis=1
)

df["capex_label"] = df["capex_intensity"].apply(
    capex_label
)

df["fcf_conversion"] = df.apply(
    lambda x: calculate_fcf_conversion(
        x.free_cash_flow,
        x.operating_profit
    ),
    axis=1
)

df["capital_pattern"] = df.apply(
    lambda x: capital_allocation_pattern(
        x.operating_activity,
        x.investing_activity,
        x.financing_activity,
        x.cfo_quality
    ),
    axis=1
)

In [6]:
df.head()

,company_id,year,sales,operating_profit,net_profit,operating_activity,investing_activity,financing_activity,free_cash_flow,cfo_quality,capex_intensity,capex_label,fcf_conversion,capital_pattern
0,ABB,2012-12,1653.0,202.0,145.0,101.0,-59.0,-42.0,42.0,0.70,3.57,Moderate,20.79,Reinvestor
1,ABB,2014-03,2276.0,267.0,198.0,0.0,0.0,0.0,0.0,0.00,0.00,Asset Light,0.00,Cash Accumulator
2,ABB,2015-03,2289.0,312.0,229.0,-35.0,-1864.0,1902.0,-1899.0,-0.15,81.43,Capital Intensive,-608.65,Growth Funded by Debt
3,ABB,2016-03,2614.0,365.0,255.0,1544.0,-816.0,-722.0,728.0,6.05,31.22,Capital Intensive,199.45,Shareholder Returns
4,ABB,2017-03,2903.0,398.0,277.0,2189.0,-1730.0,-455.0,459.0,7.90,59.59,Capital Intensive,115.33,Shareholder Returns


In [7]:
df["capex_label"].value_counts(dropna=False)

capex_label
Capital Intensive    505
Asset Light          286
Moderate             265
Name: count, dtype: int64

In [8]:
df["capital_pattern"].value_counts(dropna=False)

capital_pattern
Shareholder Returns      422
Mixed                    205
Reinvestor               165
Growth Funded by Debt     98
Liquidating Assets        95
Distress Signal           41
Unclassified              15
Pre-Revenue               10
Cash Accumulator           5
Name: count, dtype: int64

In [9]:
df[
    df["capital_pattern"] == "Unclassified"
][[
    "company_id",
    "year",
    "operating_activity",
    "investing_activity",
    "financing_activity",
    "cfo_quality"
]]

,company_id,year,operating_activity,investing_activity,financing_activity,cfo_quality
185,BEL,2013-03,-1539.0,1753.0,-158.0,-1.69
186,BEL,2014-03,-569.0,759.0,-213.0,-0.60
189,BEL,2017-03,-61.0,3115.0,-2857.0,-0.04
203,BHEL,2019-03,-3860.0,1919.0,-32.0,-3.85
401,HAL,2017-03,-423.0,3021.0,-164.0,-0.16
402,HAL,2018-03,-734.0,594.0,-2540.0,-0.37
506,ICICIGI,2015-03,-98.0,171.0,-93.0,-0.17
516,ICICIPRULI,2013-03,-2641.0,1564.0,-514.0,-1.74
517,ICICIPRULI,2014-03,-2382.0,6012.0,-1091.0,-1.53
527,ICICIPRULI,2024-03,-7315.0,7420.0,-88.0,-8.60


### Generating capital_allocation.csv

In [10]:
df["cfo_sign"] = df["operating_activity"].apply(
    lambda x: "+" if x >= 0 else "-"
)

df["cfi_sign"] = df["investing_activity"].apply(
    lambda x: "+" if x >= 0 else "-"
)

df["cff_sign"] = df["financing_activity"].apply(
    lambda x: "+" if x >= 0 else "-"
)

In [11]:
capital_allocation = df[
    [
        "company_id",
        "year",
        "cfo_sign",
        "cfi_sign",
        "cff_sign",
        "capital_pattern",
    ]
].copy()

In [12]:
capital_allocation.rename(
    columns={
        "capital_pattern": "pattern_label"
    },
    inplace=True,
)

In [13]:
from pathlib import Path

Path("../output").mkdir(
    exist_ok=True
)

In [14]:
capital_allocation.to_csv(
    "../output/capital_allocation.csv",
    index=False
)

In [15]:
capital_allocation.head()

,company_id,year,cfo_sign,cfi_sign,cff_sign,pattern_label
0,ABB,2012-12,+,-,-,Reinvestor
1,ABB,2014-03,+,+,+,Cash Accumulator
2,ABB,2015-03,-,-,+,Growth Funded by Debt
3,ABB,2016-03,+,-,-,Shareholder Returns
4,ABB,2017-03,+,-,-,Shareholder Returns


In [16]:
print(capital_allocation.shape)

(1056, 6)
